In [4]:
import pdfplumber
import os
import pandas as pd
import re

In [5]:
def year_month_key(filename):
    year, month = map(int, re.search(r"Summary_(\d+)_(\d+)_", filename).groups())
    return year, month

def merge_pdfs_to_single_txt(source_folder, output_txt_name):

    file_names = [x for x in os.listdir("Reports") if x.startswith("Summary_")]
    file_names = sorted(file_names, key=year_month_key)

    global df 

    df = pd.DataFrame(columns = ["ReportName", "Text"])
    index = 0

    for file_name in file_names:
        pdf_path = os.path.join(source_folder, file_name)
        
        try:
            with pdfplumber.open(pdf_path) as pdf:
                file_content = []
                for page in pdf.pages:
                    text = page.extract_text()
                    if text:
                        file_content.append(text)
                single_doc_text = "\n".join(file_content)
                df.loc[index] = [file_name, single_doc_text]
                index+=1
                print(f"Okundu: {file_name}")
                
        except Exception as e:
            print(f"Hata oluştu ({file_name}): {e}")

    df.to_excel("1_SummaryReportsV1.xlsx")
    print(f"\nBütün işlemler tamamlandı! Dosya konumu: {output_txt_name}")

merge_pdfs_to_single_txt("Reports", "Tum_Raporlar.txt")

Okundu: Summary_2006_1_January.pdf
Okundu: Summary_2006_2_February.pdf
Okundu: Summary_2006_3_March.pdf
Okundu: Summary_2006_4_April.pdf
Okundu: Summary_2006_5_May.pdf
Okundu: Summary_2006_6_June.pdf
Okundu: Summary_2006_7_June.pdf
Okundu: Summary_2006_8_July.pdf
Okundu: Summary_2006_9_August.pdf
Okundu: Summary_2006_10_September.pdf
Okundu: Summary_2006_11_October.pdf
Okundu: Summary_2006_12_November.pdf
Okundu: Summary_2006_13_December.pdf
Okundu: Summary_2007_1_January.pdf
Okundu: Summary_2007_2_February.pdf
Okundu: Summary_2007_3_March.pdf
Okundu: Summary_2007_4_April.pdf
Okundu: Summary_2007_5_May.pdf
Okundu: Summary_2007_6_June.pdf
Okundu: Summary_2007_7_July.pdf
Okundu: Summary_2007_8_August.pdf
Okundu: Summary_2007_9_September.pdf
Okundu: Summary_2007_10_October.pdf
Okundu: Summary_2007_11_November.pdf
Okundu: Summary_2007_12_December.pdf
Okundu: Summary_2008_1_January.pdf
Okundu: Summary_2008_2_February.pdf
Okundu: Summary_2008_3_March.pdf
Okundu: Summary_2008_4_April.pdf
Okun

In [6]:
df.head(5)

,ReportName,Text
0,Summary_2006_1_January.pdf,No: 2006-6\n27 January 2006\nSUMMARY OF THE MO...
1,Summary_2006_2_February.pdf,Number: 2006- 10 7 March 2006\nSUMMARY OF THE ...
2,Summary_2006_3_March.pdf,"No: 2006-15\nMarch 31, 2006\nSUMMARY OF THE MO..."
3,Summary_2006_4_April.pdf,"No: 2006-21\nMay 04, 2006\nSUMMARY OF THE MONE..."
4,Summary_2006_5_May.pdf,"No: 2006-24\nJune 02, 2006\nSUMMARY OF THE MON..."


In [7]:
import nltk
import re

nltk.download('punkt')

SentenceDf = pd.DataFrame(columns = ["ReportName", "Sentences"])

def sentences_from_file(df):

    with open("titles.txt", "r", encoding="utf-8") as f:
        titles = [line.strip() for line in f if line.strip()]
    index = 0
    
    for row in df.index:

        reportname = df.loc[row, "ReportName"]
        full_text = df.loc[row, "Text"]
        clean_text = full_text.replace('\n', ' ')
        for title in titles:
            clean_text = clean_text.replace(title, ".")
        sentences = nltk.sent_tokenize(clean_text)
        
        for sentence in sentences:
            if any(item in sentence for item in titles) or len(sentence) <22 or sentence[:3] == "No:" or "SUMMARY OF THE MONETARY POLICY MEETING" in sentence or "cid:3" in sentence or "Meeting Date:" in sentence or re.match(r"^\d+\s", sentence) :
                continue

            sentence = re.sub(r"^\d+\.\s*", "", sentence)
            sentence = re.sub(r"^\d+\s+(?=[A-Z])", "", sentence)
            SentenceDf.loc[index] = [reportname, sentence]
            index+=1

    print(f"Başarılı! Toplam {index} cümle kaydedildi.")
        
sentences_from_file(df)

SentenceDf.to_excel("1_SummaryReportsV2.xlsx")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\muham\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Başarılı! Toplam 19167 cümle kaydedildi.


In [8]:
SentenceDf.head()

,ReportName,Sentences
0,Summary_2006_1_January.pdf,"In December, significant increases in the pric..."
1,Summary_2006_1_January.pdf,"On the other hand, fruit and vegetable prices,..."
2,Summary_2006_1_January.pdf,"With 7.72 percent, annual inflation was realiz..."
3,Summary_2006_1_January.pdf,"VAT cuts made in food, health and education se..."
4,Summary_2006_1_January.pdf,Even if the downward course in the inflation t...


In [9]:
len(SentenceDf)

19167

In [10]:
summary = (
    SentenceDf
    .groupby("Sentences")
    .agg(
        count=("ReportName", "size"),
        reports=("ReportName", lambda s: sorted(set(s)))
    )
    .reset_index()
    .sort_values("count", ascending=False)
)

summary[summary["count"] > 9].head(3)

summary.to_excel("1_RepeatedSentences.xlsx")

In [11]:
SentenceDf2 = SentenceDf.drop_duplicates(subset="Sentences")
SentenceDf2.to_excel("1_SummaryReportsV3.xlsx")
print(len(SentenceDf2))

16817


In [12]:
SentenceDf2.head(3)

,ReportName,Sentences
0,Summary_2006_1_January.pdf,"In December, significant increases in the pric..."
1,Summary_2006_1_January.pdf,"On the other hand, fruit and vegetable prices,..."
2,Summary_2006_1_January.pdf,"With 7.72 percent, annual inflation was realiz..."


In [13]:
with open("1_SummaryReportsV3.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(SentenceDf2["Sentences"].astype(str)))